# Stage 32A: TCN uncertainty profile lock

Stage 31Aで事前定義済みの4候補を保存済みcut reportだけで完全監査します。CPUで実行でき、再学習・再推論・予約120 wellsの使用・提出作成はありません。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
print('Stage 32A is CPU-only; a GPU runtime is unnecessary.')


In [ ]:
stage31a_run = artifact_dir / 'stage31a_tcn_uncertainty_full_v001'
required = [stage31a_run / 'summary.json', stage31a_run / 'uncertainty_cut_report.parquet']
for path in required:
    assert path.is_file(), f'Missing Stage 31A artifact: {path}'
stage31 = json.loads((stage31a_run / 'summary.json').read_text())
assert stage31['stage31a_complete'] is True
assert stage31['reserved_confirmation_used'] is False
print('Stage 31A report is ready:', stage31a_run)


In [ ]:
RUN_ID = 'stage32a_tcn_profile_lock_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    result = subprocess.run([
        'uv', 'run', 'rogii-residual-profile-lock',
        '--config', 'configs/experiment/stage32a_tcn_profile_lock.yaml',
        '--stage31a-run', str(stage31a_run),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ], cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f'Stage 32A failed with exit code {result.returncode}')
else:
    print('Reusing completed run:', run_dir)


In [ ]:
import pandas as pd

summary = json.loads((run_dir / 'summary.json').read_text())
display(pd.DataFrame(summary['profile_audits']).sort_values('candidate_rmse'))
{key: summary[key] for key in [
    'stage32a_complete',
    'promoted_to_stage32b_reserved_confirmation',
    'profiles_audited',
    'eligible_profiles',
    'selected_profile',
    'reserved_confirmation_used',
    'next_step',
]}
